In [136]:
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import multiprocessing
import time
from qutip import *
from qiskit.circuit.library import CRXGate
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit import IfElseOp
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.quantum_info import state_fidelity, Statevector
from qiskit_experiments.library import StateTomography
from math import pi

# ----------------------------
# Connect to IBM Quantum Service
# ----------------------------
service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/c9a9b9d7c5bd452daa3229a9a4dc4056:d3802968-f622-4b53-94c7-f99f2ccb081b::',
    token='4OFBI96RqSuBi4vozv8sQ1lic25Ko6s9XA9tYUtN7Xle',
)
# ----------------------------
# Backend setup
# ----------------------------


# Get all CPU cores for parallelization
num_cores = multiprocessing.cpu_count()
print(f"Using {num_cores} CPU threads for simulation")
backend = service.backend("ibm_torino")  # replace with your desired backend name
# Configure the Aer simulator for max performance
backend = AerSimulator(
    method='statevector',            # fastest method for most circuits
    max_parallel_threads=num_cores,  # use all cores
    max_parallel_experiments=num_cores,  # parallelize experiments if possible
    blocking_enable=False,           # non-blocking execution
    precision='double'               # you can set 'single' to reduce memory & boost speed
).from_backend(backend)


backend = service.backend("ibm_torino")  # replace with your desired backend name
# ----------------------------
# Sampler setup
# ----------------------------
sampler = SamplerV2(mode=backend)
sampler.options.experimental = {"execution_path": "gen3-experimental"}

# ----------------------------
# Preset transpiler pass manager
# ----------------------------
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    backend=backend
)


qiskit_runtime_service._discover_account:WARNING:2025-10-20 22:57:23,608: Loading account with the given token. A saved account will not be used.


Using 14 CPU threads for simulation


In [143]:
# ----------------------------
# Unitary construction function
# ----------------------------
def M_Unitary(kappa):
    """Return a UnitaryGate generated from the given coupling parameter kappa."""
    H = kappa * (tensor(sigmam(), sigmap()) + tensor(sigmap(), sigmam()))
    M_Unitary = (-1j * H).expm()
    return UnitaryGate(M_Unitary.full())


In [144]:
def remote_cx(qc,control, target,commA, commB, ENA, ENB, creg, creg_index, kappa_Fiber, Steps, kappa_Transductor):
    qc.barrier()
    qc.h(commA)
    qc.cx(commA, commB)
    qc.barrier()
    qc.append(M_Unitary(kappa_Transductor), [commA, ENA])
    qc.append(M_Unitary(kappa_Transductor), [commB, ENB])
    qc.reset(ENA)
    qc.reset(ENB)
    qc.append(M_Unitary(kappa_Fiber), [commA, ENA])
    qc.append(M_Unitary(kappa_Fiber), [commB, ENB])
    for i in range(Steps):
        qc.reset(ENA)
        qc.reset(ENB)
        qc.append(M_Unitary(kappa_Fiber), [commA, ENA])
        qc.append(M_Unitary(kappa_Fiber), [commB, ENB])
    qc.cx(control, commA)
    qc.measure(commA, creg[creg_index])
    with qc.if_test((creg[creg_index], 1)):
        qc.x(commA)
        qc.x(commB)
    qc.cx(commB, target)
    qc.h(commB)
    qc.measure(commB, creg[creg_index + 1])
    with qc.if_test((creg[creg_index + 1], 1)):
        qc.z(control)
    qc.reset(commA)
    qc.reset(commB)
    #qc.barrier()




In [145]:
Layout_Topologies = {"Line": {      "A": {'commA1': 10, 'commA2': 17, 'ENA': 8,  'PA1': 9},
                                    "B": {'commB1': 11, 'commB2': 18, 'ENB': 13, 'PB1': 12},
                                    "C": {'commC1': 27, 'commC2': 29, 'ENC': 36, 'PC1': 28},
                                    "D": {'commD1': 30, 'commD2': 31, 'END': 33, 'PD1': 32}
},
                "Ring": {
                                    "A": {'commA1': 10, 'commA2': 17, 'ENA': 8,  'PA1': 9},
                                    "B": {'commB1': 11, 'commB2': 18, 'ENB': 12, 'PB1': 13},
                                    "C": {'commC1': 27, 'commC2': 29, 'ENC': 36, 'PC1': 28},
                                    "D": {'commD1': 30, 'commD2': 31, 'END': 33, 'PD1': 32}
                },
                "Star": {
                                    "A": {'commA1': 17, 'ENA': 8, 'PA1': 9},
                                    "B": {'commB1': 27, 'commB2': 31, 'commB3': 46, 'commB4': 50,  'ENB1': 29, 'ENB2': 48,  'PB1': 36},
                                    "C": {'commC1': 18, 'ENC': 11, 'PC1': 12},
                                    "D": {'commD1': 55, 'END': 64, 'PD1': 66}
}}

In [146]:

# assumes q is an indexable sequence with at least 16 elements

def circuit_GHZ(Topology, Steps, kappa_Fiber, kappa_Transductor):
    q = QuantumRegister(16, 'q')
    c = ClassicalRegister(16, 'c')
    qc = QuantumCircuit(q, c)
    if Topology in ("Line", "Ring"):
        initial_layout = [
            # --- A ---
            Layout_Topologies[Topology]["A"]["ENA"],
            Layout_Topologies[Topology]["A"]["PA1"],
            Layout_Topologies[Topology]["A"]["commA1"],
            Layout_Topologies[Topology]["A"]["commA2"],

            # --- B ---
            Layout_Topologies[Topology]["B"]["ENB"],
            Layout_Topologies[Topology]["B"]["PB1"],
            Layout_Topologies[Topology]["B"]["commB1"],
            Layout_Topologies[Topology]["B"]["commB2"],

            # --- C ---
            Layout_Topologies[Topology]["C"]["ENC"],
            Layout_Topologies[Topology]["C"]["PC1"],
            Layout_Topologies[Topology]["C"]["commC1"],
            Layout_Topologies[Topology]["C"]["commC2"],

            # --- D ---
            Layout_Topologies[Topology]["D"]["END"],
            Layout_Topologies[Topology]["D"]["PD1"],
            Layout_Topologies[Topology]["D"]["commD1"],
            Layout_Topologies[Topology]["D"]["commD2"]
        ]

    elif Topology == "Star":
        initial_layout = {
            # --- A ---
            q[0]:  Layout_Topologies["Star"]["A"]["ENA"],
            q[1]:  Layout_Topologies["Star"]["A"]["PA1"],
            q[2]:  Layout_Topologies["Star"]["A"]["commA1"],  # (no commA2 in Star)

            # --- B --- (ENB split into ENB1 & ENB2; extra commB3 & commB4)
            q[3]:  Layout_Topologies["Star"]["B"]["ENB1"],
            q[4]:  Layout_Topologies["Star"]["B"]["ENB2"],
            q[5]:  Layout_Topologies["Star"]["B"]["PB1"],
            q[6]:  Layout_Topologies["Star"]["B"]["commB1"],
            q[7]:  Layout_Topologies["Star"]["B"]["commB2"],
            q[8]:  Layout_Topologies["Star"]["B"]["commB3"],
            q[9]:  Layout_Topologies["Star"]["B"]["commB4"],

            # --- C ---
            q[10]: Layout_Topologies["Star"]["C"]["ENC"],
            q[11]: Layout_Topologies["Star"]["C"]["PC1"],
            q[12]: Layout_Topologies["Star"]["C"]["commC1"],  # (no commC2 in Star)

            # --- D ---
            q[13]: Layout_Topologies["Star"]["D"]["END"],
            q[14]: Layout_Topologies["Star"]["D"]["PD1"],
            q[15]: Layout_Topologies["Star"]["D"]["commD1"],  # (no commD2 in Star)
        }

    else:
        raise ValueError(f"Unknown Topology: {Topology}")
    
    if Topology == "Ring":
        q = QuantumRegister(16, 'q')
        c = ClassicalRegister(16, 'c')
        qc = QuantumCircuit(q, c)
        qc.h(q[1])
        remote_cx(qc,control = q[1], target = q[5] , commA = q[2], commB = q[6], ENA = q[0], ENB = q[4],    creg = c, creg_index = 0, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[1], target = q[13] , commA = q[2], commB = q[15], ENA = q[4], ENB = q[12],    creg = c, creg_index = 1, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[1], target = q[9] , commA = q[3], commB = q[10], ENA = q[0], ENB = q[12],    creg = c, creg_index = 2, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)




    elif Topology == "Line":
        q = QuantumRegister(16, 'q')
        c = ClassicalRegister(16, 'c')
        qc = QuantumCircuit(q, c)
        qc.h(q[1])
        remote_cx(qc,control = q[1], target = q[5] , commA = q[2], commB = q[6], ENA = q[0], ENB = q[4],    creg = c, creg_index = 0, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[1], target = q[13] , commA = q[2], commB = q[15], ENA = q[4], ENB = q[12],    creg = c, creg_index = 1, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[1], target = q[9] , commA = q[2], commB = q[11], ENA = q[0], ENB = q[12],    creg = c, creg_index = 2, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)

    elif Topology == "Star":
        q = QuantumRegister(16, 'q')
        c = ClassicalRegister(16, 'c')
        qc = QuantumCircuit(q, c)
        qc.h(q[5])
        remote_cx(qc,control = q[5], target = q[1] , commA = q[6], commB = q[2], ENA = q[3], ENB = q[0],    creg = c, creg_index = 0, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[5], target = q[14] , commA = q[8], commB = q[15], ENA = q[4], ENB = q[13],    creg = c, creg_index = 1, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)
        remote_cx(qc,control = q[5], target = q[11] , commA = q[7], commB = q[12], ENA = q[6], ENB = q[10],    creg = c, creg_index = 2, kappa_Fiber = kappa_Fiber, Steps = Steps, kappa_Transductor = kappa_Transductor)

        


    return qc, initial_layout

In [147]:
from qiskit import transpile

Topology = "Ring"  # "Line", "Ring", or "Star"
kappa_Fiber = 0.0392
kappa_Transductor = 0.5
steps = 10
# Define registers and circuit
q = QuantumRegister(4, 'q')
c = ClassicalRegister(4, 'c')
qc = QuantumCircuit(q, c)
qc.h(q[0])
qc.cx(q[0], q[1])
qc.cx(q[0], q[2])
qc.cx(q[0], q[3])
psi = Statevector.from_instruction(qc)
exp = StateTomography(qc, backend=backend)
job = exp.run(backend=backend).block_for_results()
jobs = job.jobs()
for j in jobs:
    print("Job ID:", j.job_id())
rho = job.analysis_results("state").value 
fidelity = state_fidelity(rho, psi)
print(f"Fidelity: {fidelity:.4f}")
for i in range(steps):
    qc, initial_layout = circuit_GHZ(Topology = Topology, Steps = i, kappa_Fiber = kappa_Fiber, kappa_Transductor = kappa_Transductor)
    qc_transpiled = transpile(qc, backend=backend, initial_layout = initial_layout, optimization_level=0, layout_method='trivial', routing_method='sabre')
    exp = StateTomography(qc_transpiled, backend=backend, measurement_indices = [1, 5, 9, 13])
    job = exp.run(backend=backend).block_for_results()
    jobs = job.jobs()
    for j in jobs:
        print("Job ID:", j.job_id())
    rho = job.analysis_results("state").value 
    fidelity = state_fidelity(rho, psi)
    print(f"Steps: {i}, Fidelity: {fidelity:.4f}")

KeyboardInterrupt: 

In [155]:
kappa_Fiber = 0.0392
kappa_Transductor = 0.5
steps = 10
# Define registers and circuit
q = QuantumRegister(2, 'q')
c = ClassicalRegister(2, 'c')
qc = QuantumCircuit(q, c)
#qc.x(q[0])
qc.cx(q[0], q[1])
psi = Statevector.from_instruction(qc)
exp = StateTomography(qc, backend=backend, measurement_indices = [0, 1])
job = exp.run(sampler = sampler).block_for_results()
rho  = job.analysis_results("state").value 
jobs = job.jobs()
for j in jobs:
    print("Job ID:", j.job_id())
fidelity = state_fidelity(rho, psi)
print(f"Fidelity: {fidelity:.4f}")
for i in range(steps):
    q = QuantumRegister(6)
    c = ClassicalRegister(2)
    qc = QuantumCircuit(q, c)
    #qc.x(q[0])
    remote_cx(qc = qc,control =q[0], target = q[5], commA = q[2], commB = q[3], ENA = q[1], ENB = q[4], creg = c, creg_index = 0, kappa_Fiber = kappa_Fiber, Steps = i, kappa_Transductor = kappa_Transductor)
    exp = StateTomography(qc, backend=backend, measurement_indices = [0, 5])
    job = exp.run(backend = backend).block_for_results()
    jobs = job.jobs()
    for j in jobs:
        print("Job ID:", j.job_id())
    rho = job.analysis_results("state").value 
    fidelity = state_fidelity(rho, psi)
    print(f"Steps: {i}, Fidelity: {fidelity:.4f}")
    

/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:13: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho  = job.analysis_results("state").value


Job ID: d3rb8arld2is73fhr7og
Fidelity: 0.8599
Job ID: d3rb8ndq5lhs73bdjqc0
Steps: 0, Fidelity: 0.6855


/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:30: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho = job.analysis_results("state").value


Job ID: d3rb93rnquss73e7g660
Steps: 1, Fidelity: 0.6728


/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:30: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho = job.analysis_results("state").value


...

/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:30: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho = job.analysis_results("state").value


Job ID: d3rbbu5q5lhs73bdjtbg
Steps: 8, Fidelity: 0.5409


/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:30: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho = job.analysis_results("state").value


Job ID: d3rbcbdq5lhs73bdjtog
Steps: 9, Fidelity: 0.5343


/var/folders/40/8bqwdxj16852wcwh705klppw0000gn/T/ipykernel_1078/1946209498.py:30: DeprecationWarning: Leaving `dataframe` unset or setting it to `False` for `ExperimentData.analysis_results` is deprecated as of qiskit-experiments 0.9.0. Future releases may change the default to `True` and remove the option to set the value to `False`.
  rho = job.analysis_results("state").value
